# Milestone 1: Foundations + Task 1 (Universal AE) + Task 2 (Hard Routing)
Deadline: October 4, 2026
Runtime Environment: Google Colab (T4 GPU recommended)

In [ ]:
# 1. Mount Google Drive & Clone Repository
# Set YOUR_GITHUB_USERNAME below or rely on GH_TOKEN secret in Colab Secrets
import os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/genai_assignment', exist_ok=True)

try:
    token = userdata.get('GH_TOKEN')
    # Replace YOUR_GITHUB_USERNAME with your GitHub username if needed
    username = os.environ.get('GH_USER', '50022005')
    repo_url = f"https://{token}@github.com/{username}/restoration-studio.git"
except Exception as e:
    print(f"[WARN] Could not load GH_TOKEN secret ({e}). Set repo_url manually if needed.")
    repo_url = "https://github.com/YOUR_GITHUB_USERNAME/restoration-studio.git"

repo_dir = Path("/content/restoration-studio")
if not repo_dir.exists():
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

%cd /content/restoration-studio
print('✓ Repository cloned & working directory set to:', os.getcwd())

In [ ]:
# 2. Install dependencies (runs inside repository root)
!pip install -q -r requirements.txt

In [ ]:
# 3. Run Oxford-IIIT Pet Dataset Preparation & Manifest Generation
!python data/prepare_pets.py \
    --raw_dir /content/drive/MyDrive/genai_assignment/data/pets_raw \
    --cache_dir /content/drive/MyDrive/genai_assignment/data/pets_cache \
    --manifest_dir data/manifests \
    --seed 42

In [ ]:
# 4. Task 1: Train Universal Autoencoder (Optuna + Final Training)
!python src/task1/train.py --config configs/task1.yaml

In [ ]:
# 5. Task 1: Evaluate Universal Autoencoder
!python src/task1/evaluate.py --config configs/task1.yaml

In [ ]:
# 6. Task 1: Export ONNX Model & Verify Parity
!python src/task1/export_onnx.py --config configs/task1.yaml

In [ ]:
# 7. Task 2: Train Corruption Classifier
!python src/task2/train_classifier.py --config configs/task2.yaml

In [ ]:
# 8. Task 2: Train 3 Specialist Autoencoders
!python src/task2/train_specialists.py --config configs/task2.yaml

In [ ]:
# 9. Task 2: Evaluate Hard Routing (Oracle vs Predicted)
!python src/task2/evaluate.py --config configs/task2.yaml

In [ ]:
# 10. Task 2: Export All ONNX Models (Classifier + 3 Specialists)
!python src/task2/export_onnx.py --config configs/task2.yaml